# Auditoría de Videos Publicitarios con Vertex AI y BigQuery

Este notebook guía una evaluación paso a paso para auditar videos publicitarios con Vertex AI y guardar los resultados en BigQuery.


## Paso 1) Instalación de dependencias
Ejecuta esta celda una sola vez por sesión.


In [ ]:
# Instalación de librerías necesarias
!pip install -q google-cloud-bigquery google-cloud-storage google-cloud-aiplatform ipywidgets pandas


## Paso 1.5) Autenticación con Google (opcional pero recomendado)
Si vas a usar Vertex AI o BigQuery desde Colab, autentícate.


In [ ]:
# Autenticación (si estás en Colab)
try:
    from google.colab import auth
    auth.authenticate_user()
    print("Autenticación completada.")
except Exception as exc:
    print("No se pudo iniciar autenticación automática. Continúa si ya estás autenticado.", exc)


## Paso 2) Formulario de captura de datos
Completa los campos y selecciona las características a evaluar.


In [ ]:
import json
import datetime
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML
from google.cloud import bigquery, aiplatform
import pandas as pd

# --- Configuración de criterios ---
criteria_catalog = {
    "Calidad visual": [
        {"id": "imagenes_referentes", "label": "Imágenes referentes al propósito (Sí/No)"},
        {"id": "ruptura_patron", "label": "Ruptura del patrón inicial (Cumple/No cumple)"},
        {"id": "lead_in", "label": "Lead-in instantáneo (Cumple/No cumple)"},
        {"id": "velocidad_lectura", "label": "Velocidad de lectura (Alta/Media/Baja)"},
        {"id": "dinamizacion_producto", "label": "Dinamización de producto (Cumple/No cumple)"},
        {"id": "jerarquia_tamano", "label": "Jerarquía de tamaño (Cumple/No cumple)"},
        {"id": "sincronia_audio_visual", "label": "Sincronía audio-visual (Cumple/No cumple)"},
        {"id": "legibilidad_movil", "label": "Legibilidad en móvil (Cumple/No cumple)"},
        {"id": "identificacion_precoz", "label": "Identificación precoz (Cumple/No cumple)"},
        {"id": "persistencia_identidad", "label": "Persistencia de identidad (Cumple/No cumple)"},
        {"id": "cierre_marca", "label": "Cierre de marca (Cumple/No cumple)"},
        {"id": "tangibilidad_beneficio", "label": "Tangibilidad del beneficio (Cumple/No cumple)"},
        {"id": "resolucion_barreras", "label": "Resolución de barreras (Cumple/No cumple)"},
        {"id": "uso_producto", "label": "Uso de producto/servicio (Cumple/No cumple)"},
        {"id": "independencia_audio", "label": "Independencia del audio (Cumple/No cumple)"},
        {"id": "respeto_zonas", "label": "Respeto de zonas seguras (Cumple/No cumple)"},
        {"id": "contraste", "label": "Contraste fondo-figura (Cumple/No cumple)"},
    ],
    "Creatividad y branding": [
        {"id": "singularidad_mensaje", "label": "Singularidad del mensaje (Cumple/No cumple)"},
        {"id": "hook_visual", "label": "Optimización del hook visual (Alta/Media/Baja)"},
        {"id": "rebalanceo", "label": "Rebalanceo branding-producto (Alta/Media/Baja)"},
        {"id": "saturacion_textual", "label": "Gestión de saturación textual (Baja/Media/Alta)"},
    ],
}

criteria_options = []
for category, items in criteria_catalog.items():
    for item in items:
        criteria_options.append(f"[{category}] {item['label']}")

# --- Campos principales ---
video_url = widgets.Text(description="Video URL", placeholder="https://www.youtube.com/watch?v=...", layout=widgets.Layout(width='100%'))
video_name = widgets.Text(description="Video name", layout=widgets.Layout(width='100%'))
brand_name = widgets.Text(description="Brand name", layout=widgets.Layout(width='100%'))
product_name = widgets.Text(description="Product name", layout=widgets.Layout(width='100%'))
product_category = widgets.Text(description="Product category", layout=widgets.Layout(width='100%'))
campaign_name = widgets.Text(description="Campaign name", layout=widgets.Layout(width='100%'))
target_audience = widgets.Text(description="Target audience", layout=widgets.Layout(width='100%'))
market = widgets.Text(description="Market", layout=widgets.Layout(width='100%'))
additional_context = widgets.Textarea(description="Context", placeholder="Notas adicionales para el análisis", layout=widgets.Layout(width='100%', height='80px'))

project_id = widgets.Text(description="Project ID", layout=widgets.Layout(width='100%'))
project_zone = widgets.Text(description="Project zone", value="us-central1", layout=widgets.Layout(width='100%'))
llm_name = widgets.Text(description="Model", value="text-bison@002", layout=widgets.Layout(width='100%'))

temperature = widgets.FloatSlider(description="Temperature", value=0.2, min=0, max=1, step=0.05)
max_output_tokens = widgets.IntSlider(description="Max tokens", value=2048, min=256, max=4096, step=128)
top_p = widgets.FloatSlider(description="Top-p", value=0.95, min=0.1, max=1, step=0.05)
top_k = widgets.IntSlider(description="Top-k", value=40, min=0, max=100, step=5)

bq_dataset_name = widgets.Text(description="BQ dataset", layout=widgets.Layout(width='100%'))
bq_table_name = widgets.Text(description="BQ table", layout=widgets.Layout(width='100%'))

criteria_selector = widgets.SelectMultiple(
    options=criteria_options,
    value=tuple(criteria_options),
    description="Criterios",
    layout=widgets.Layout(width='100%', height='240px')
)

# --- Layout ---
metadata_box = widgets.VBox([
    video_url, video_name, brand_name, product_name, product_category, campaign_name, target_audience, market, additional_context
])

vertex_box = widgets.VBox([
    project_id, project_zone, llm_name, temperature, max_output_tokens, top_p, top_k
])

bq_box = widgets.VBox([
    bq_dataset_name, bq_table_name
])

form_grid = widgets.GridBox(
    children=[metadata_box, vertex_box, bq_box],
    layout=widgets.Layout(grid_template_columns='repeat(3, 1fr)', grid_gap='20px')
)

display(HTML("<h3>Datos generales</h3>"))
display(metadata_box)

display(HTML("<h3>Configuración Vertex AI</h3>"))
display(vertex_box)

display(HTML("<h3>Configuración BigQuery</h3>"))
display(bq_box)

display(HTML("<h3>Características a evaluar</h3>"))
display(criteria_selector)



## Paso 3) Prompt completo (vista previa)
Genera el prompt usando los datos del formulario.


In [ ]:
prompt_output = widgets.Textarea(layout=widgets.Layout(width='100%', height='360px'))
refresh_prompt_button = widgets.Button(description="Actualizar prompt", button_style='info')

criteria_definitions = {
    "imagenes_referentes": "Imágenes referentes al propósito (Sí/No): Las imágenes mostradas deben estar alineadas con la promoción o producto anunciado, especialmente en los primeros 3 segundos.",
    "ruptura_patron": "Ruptura del patrón inicial (Cumple/No cumple): Ocurre un cambio visual significativo (corte, animación, color) en los primeros 0-3 segundos.",
    "lead_in": "Lead-in instantáneo (Cumple/No cumple): El sonido comienza en el 00:00 sin silencios iniciales.",
    "velocidad_lectura": "Velocidad de lectura (Alta/Media/Baja): La duración de las escenas permite leer el texto completo cómodamente.",
    "dinamizacion_producto": "Dinamización de producto (Cumple/No cumple): Uso de técnicas cinematográficas y de edición para mantener la atención sostenida y evitar la fatiga visual.",
    "jerarquia_tamano": "Jerarquía de tamaño (Cumple/No cumple): El elemento más importante (%, precio, producto) es el objeto más grande en pantalla.",
    "sincronia_audio_visual": "Sincronía audio-visual (Cumple/No cumple): La locución verbal coincide con la información clave que aparece en texto.",
    "legibilidad_movil": "Legibilidad en móvil (Cumple/No cumple): Los textos principales son lo suficientemente grandes para verse en una pantalla de celular.",
    "identificacion_precoz": "Identificación precoz (Cumple/No cumple): Logo/color corporativo aparece en los primeros 5 segundos.",
    "persistencia_identidad": "Persistencia de identidad (Cumple/No cumple): Elementos de marca (logo, marco, marca de agua) visibles el 100% del tiempo.",
    "cierre_marca": "Cierre de marca (Cumple/No cumple): El video termina con una placa final de logo/slogan.",
    "tangibilidad_beneficio": "Tangibilidad del beneficio (Cumple/No cumple): El beneficio es concreto (%, $, cantidad) y no abstracto.",
    "resolucion_barreras": "Resolución de barreras (Cumple/No cumple): Los legales/condiciones son visualmente secundarios al beneficio principal.",
    "uso_producto": "Uso de producto/servicio (Cumple/No cumple): Se muestra el producto o servicio como protagonista visual.",
    "independencia_audio": "Independencia del audio (Cumple/No cumple): Se entiende la oferta y la marca si el video está en silencio.",
    "respeto_zonas": "Respeto de zonas seguras (Cumple/No cumple): Textos/logos evitan los bordes extremos (interfaz de UI).",
    "contraste": "Contraste fondo-figura (Cumple/No cumple): Existe alto contraste para facilitar la lectura rápida.",
    "singularidad_mensaje": "Singularidad del mensaje (Cumple/No cumple): Cada escena presenta una única idea principal, sin mezclar ofertas distintas.",
    "hook_visual": "Optimización del hook visual (Alta/Media/Baja): Capacidad del video para detener el scroll mediante un estímulo visual de alto impacto en los primeros 1.5 a 3 segundos.",
    "rebalanceo": "Rebalanceo branding-producto (Alta/Media/Baja): Distribución proporcional del tiempo en pantalla entre los elementos de identidad de marca y los beneficios tangibles del producto.",
    "saturacion_textual": "Gestión de saturación textual (Baja/Media/Alta): Optimización de la carga cognitiva mediante la reducción de palabras en pantalla para facilitar el procesamiento visual.",
}

label_to_id = {}
for category, items in criteria_catalog.items():
    for item in items:
        label_to_id[f"[{category}] {item['label']}"] = item['id']


def build_prompt():
    selected_ids = [label_to_id[label] for label in criteria_selector.value]
    selected_definitions = [f"- {criteria_definitions[cid]}" for cid in selected_ids]

    return f"""Actúa como un **Auditor de creatividad, efectos visuales y branding**, con especialización en neuromarketing y performance digital. Tu objetivo es analizar videos publicitarios para obtener insights que permitan que el video sea atractivo y memorable para el usuario, sin enfocarse únicamente en la conversión.

Tu metodología es imparcial: utiliza evaluaciones técnicas con "sí/no", "cumple/no cumple" o escalas ("alta", "media", "baja") según corresponda. La salida debe organizarse en **tablas por categoría**, con resultados, justificación y sugerencias/recomendaciones, además de una puntuación numérica.

## Instrucciones
1. Analiza el video paso a paso.
2. Evalúa cada criterio según su tipo:
   - **Sí/No** → responde con "Sí" o "No".
   - **Cumple/No cumple** → responde con "Cumple" o "No cumple".
   - **Alta/Media/Baja** → responde con la escala correspondiente.
3. Para cada criterio, incluye:
   - Resultado.
   - Justificación breve.
   - Sugerencia o recomendación de mejora.
4. Cada uno de los **criterios seleccionados** tiene el mismo valor dentro del total.
5. Al final de cada categoría, asigna una **puntuación de 1 a 100** que refleje el desempeño global en esa categoría.
6. Calcula el **score final del video** sumando los resultados de los criterios seleccionados y normalizando a 100.
7. Cuando existan múltiples videos evaluados, calcula el **score global** como el **promedio de los scores finales de todos los videos analizados hasta la fecha**.
8. Entrega recomendaciones estratégicas generales basadas en los puntos débiles.
9. **Incluye metadatos del video evaluado**:
   - Hora y fecha de ejecución.
   - Video ID.
   - Video name (título del video).
   - Brand name (marca asociada al video).
   - Product name.
   - Product category.
   - Campaign name.
   - Target audience.
   - Market.

---

## Criterios de evaluación con definiciones (solo los seleccionados)
{chr(10).join(selected_definitions)}

---

## Formato de salida requerido (estricto JSON)
Devuelve **únicamente** un objeto JSON válido, sin texto adicional. Usa exactamente este esquema:

{{
  "metadata": {{
    "execution_time": "ISO-8601",
    "video_id": "...",
    "video_name": "...",
    "brand_name": "...",
    "product_name": "...",
    "product_category": "...",
    "campaign_name": "...",
    "target_audience": "...",
    "market": "..."
  }},
  "categories": [
    {{
      "category_name": "...",
      "criteria": [
        {{
          "criterion_id": "...",
          "criterion_name": "...",
          "result": "Sí|No|Cumple|No cumple|Alta|Media|Baja",
          "justification": "...",
          "recommendation": "..."
        }}
      ],
      "category_score": 0
    }}
  ],
  "final_score": 0,
  "global_score": 0,
  "strategic_recommendations": ["..."]
}}

---

## Datos del video
Video URL: {video_url.value}
Video name: {video_name.value}
Brand name: {brand_name.value}
Product name: {product_name.value}
Product category: {product_category.value}
Campaign name: {campaign_name.value}
Target audience: {target_audience.value}
Market: {market.value}
Contexto adicional: {additional_context.value}
"""


def refresh_prompt(_=None):
    prompt_output.value = build_prompt()

refresh_prompt_button.on_click(refresh_prompt)
refresh_prompt()

display(refresh_prompt_button)
display(prompt_output)


## Paso 4) Envío a Vertex AI y visualización del resultado
Ejecuta la evaluación y revisa los resultados en un formato legible.


In [ ]:
run_button = widgets.Button(description="Ejecutar evaluación", button_style='success')
output_area = widgets.Output()


def call_vertex(prompt_text):
    aiplatform.init(project=project_id.value, location=project_zone.value)
    model = aiplatform.TextGenerationModel.from_pretrained(llm_name.value)
    response = model.predict(
        prompt_text,
        temperature=temperature.value,
        max_output_tokens=max_output_tokens.value,
        top_p=top_p.value,
        top_k=top_k.value,
    )
    return response.text


def render_tables(parsed):
    display(Markdown("### Metadatos"))
    meta_df = pd.DataFrame([parsed.get("metadata", {})])
    display(meta_df)

    display(Markdown("### Resultados por categoría"))
    for category in parsed.get("categories", []):
        display(Markdown(f"#### {category.get('category_name','Sin categoría')}"))
        criteria_rows = category.get("criteria", [])
        if criteria_rows:
            df = pd.DataFrame(criteria_rows)
            display(df)
        display(Markdown(f"**Score categoría:** {category.get('category_score', 0)}"))

    display(Markdown("### Scores"))
    display(Markdown(f"**Score final:** {parsed.get('final_score', 0)}"))
    display(Markdown(f"**Score global acumulado:** {parsed.get('global_score', 0)}"))

    display(Markdown("### Recomendaciones estratégicas generales"))
    for rec in parsed.get("strategic_recommendations", []):
        display(Markdown(f"- {rec}"))


def on_run_click(_):
    with output_area:
        output_area.clear_output()
        prompt_text = build_prompt()
        display(Markdown("**Enviando prompt a Vertex AI...**"))
        response_text = call_vertex(prompt_text)
        display(Markdown("**Respuesta cruda:**"))
        display(HTML(f"<pre>{response_text}</pre>"))
        try:
            parsed = json.loads(response_text)
            display(Markdown("**Respuesta parseada:**"))
            render_tables(parsed)
            output_area.parsed_result = parsed
            output_area.raw_result = response_text
        except json.JSONDecodeError as exc:
            display(Markdown(f"⚠️ No se pudo parsear el JSON: {exc}"))
            output_area.parsed_result = None
            output_area.raw_result = response_text

run_button.on_click(on_run_click)
display(run_button)
display(output_area)


## Paso 5) Envío a BigQuery
Guarda el resultado en BigQuery y confirma el estado del pipeline.


In [ ]:
send_bq_button = widgets.Button(description="Enviar a BigQuery", button_style='primary')
bq_output = widgets.Output()


def transform_to_bq(parsed, raw_text):
    metadata = (parsed or {}).get("metadata", {})
    return [{
        "execution_time": metadata.get("execution_time") or datetime.datetime.now().isoformat(),
        "video_id": metadata.get("video_id") or video_url.value.split("v=")[-1],
        "video_name": metadata.get("video_name") or video_name.value,
        "brand_name": metadata.get("brand_name") or brand_name.value,
        "product_name": metadata.get("product_name") or product_name.value,
        "product_category": metadata.get("product_category") or product_category.value,
        "campaign_name": metadata.get("campaign_name") or campaign_name.value,
        "target_audience": metadata.get("target_audience") or target_audience.value,
        "market": metadata.get("market") or market.value,
        "final_score": (parsed or {}).get("final_score"),
        "global_score": (parsed or {}).get("global_score"),
        "categories_json": json.dumps((parsed or {}).get("categories", []), ensure_ascii=False),
        "strategic_recommendations": json.dumps((parsed or {}).get("strategic_recommendations", []), ensure_ascii=False),
        "raw_response": raw_text,
    }]


def insert_into_bigquery(rows):
    client = bigquery.Client(project=project_id.value)
    table_id = f"{project_id.value}.{bq_dataset_name.value}.{bq_table_name.value}"
    errors = client.insert_rows_json(table_id, rows)
    return errors


def on_send_bq(_):
    with bq_output:
        bq_output.clear_output()
        parsed = getattr(output_area, 'parsed_result', None)
        raw_text = getattr(output_area, 'raw_result', '')
        if not raw_text:
            print("Primero ejecuta la evaluación en el Paso 4.")
            return
        rows = transform_to_bq(parsed, raw_text)
        errors = insert_into_bigquery(rows)
        if errors == []:
            print("✅ Datos insertados correctamente en BigQuery. Todos los pasos se ejecutaron correctamente.")
        else:
            print("❌ Errores al insertar:", errors)

send_bq_button.on_click(on_send_bq)
display(send_bq_button)
display(bq_output)
